# 10 经验模态分解 + LSTM (EMD-LSTM) 证券交易策略

## 论文参考

- **作者**: Yi Chen (陈益)
- **年份**: 2024
- **标题**: *Automated Improvement of Securities Trading Strategies using Machine Learning*
- **关键成果**: 年化收益率 71.85% - 127.27%

### 摘要

本文提出一种基于经验模态分解(EMD)和LSTM的混合预测方法。EMD将非平稳、非线性的
价格序列自适应分解为多个固有模态函数(IMF)和一个残差趋势项，每个IMF捕获不同时间尺度的
振荡模式。为每个IMF分量分别训练独立的LSTM模型，最后将各分量预测结果求和得到最终价格预测。
这种"分而治之"的思路显著提升了预测精度。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 经验模态分解 (EMD)

EMD是一种自适应的时频分析方法，将信号 $x(t)$ 分解为：

$$x(t) = \sum_{k=1}^{K} c_k(t) + r(t)$$

其中 $c_k(t)$ 为第 $k$ 个固有模态函数(IMF)，$r(t)$ 为残差趋势项。

### EMD 筛分过程 (Sifting)

对信号 $h(t)$ 迭代执行：

1. 找到所有局部极大值和极小值
2. 用三次样条插值得到上包络线 $e_{\max}(t)$ 和下包络线 $e_{\min}(t)$
3. 计算均值包络: $m(t) = \frac{e_{\max}(t) + e_{\min}(t)}{2}$
4. 提取候选IMF: $h(t) \leftarrow h(t) - m(t)$
5. 重复直到满足IMF条件

### IMF 条件

一个函数 $c(t)$ 是IMF当且仅当：
- 极值点数目与零交叉点数目相差不超过1
- 上下包络线关于时间轴对称（均值为零）

### 各分量独立LSTM预测

对每个分量 $c_k(t)$ 和残差 $r(t)$，训练独立LSTM：

$$\hat{c}_k(t+1) = \text{LSTM}_k(c_k(t-T+1), ..., c_k(t))$$

### 预测合成

$$\hat{x}(t+1) = \sum_{k=1}^{K} \hat{c}_k(t+1) + \hat{r}(t+1)$$

方向判断: $\text{signal} = \text{sign}(\hat{x}(t+1) - x(t))$

In [ ]:
# ============================================================
# Cell 3: 动画 - EMD 分解模式分离过程
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def animate_emd_decomposition():
    """动画展示EMD逐步将信号分解为多个IMF分量"""
    np.random.seed(42)
    t = np.linspace(0, 4 * np.pi, 500)
    # 模拟金融信号: 多频叠加
    imf1 = 0.4 * np.sin(8 * t + np.random.randn()*0.1)  # 高频振荡
    imf2 = 0.6 * np.sin(3 * t + 0.5)  # 中频波动
    imf3 = 0.8 * np.sin(0.8 * t + 1)  # 低频周期
    residual = 0.03 * t + 0.2  # 趋势
    signal = imf1 + imf2 + imf3 + residual + 0.05 * np.random.randn(len(t))

    components = [imf1, imf2, imf3, residual]
    labels = ['IMF 1 (高频振荡)', 'IMF 2 (中频波动)', 'IMF 3 (低频周期)', '残差 (Residual)']
    colors = ['#F44336', '#FF9800', '#2196F3', '#4CAF50']

    frames = []

    # Frame 0: 原始信号
    frames.append(go.Frame(
        data=[go.Scatter(x=t, y=signal, mode='lines',
                         line=dict(color='#333', width=1.5), name='原始价格序列')],
        name='原始信号'
    ))

    # 逐步提取IMF
    remaining = signal.copy()
    for step in range(len(components)):
        comp = components[step]
        remaining = remaining - comp

        data = [
            go.Scatter(x=t, y=signal, mode='lines',
                       line=dict(color='rgba(100,100,100,0.2)', width=1),
                       name='原始信号'),
        ]

        # 已提取的分量
        for j in range(step + 1):
            offset = -(j + 1) * 2.5
            data.append(go.Scatter(
                x=t, y=components[j] + offset,
                mode='lines', line=dict(color=colors[j], width=1.5),
                name=labels[j]
            ))

        # 剩余信号
        if step < len(components) - 1:
            data.append(go.Scatter(
                x=t, y=remaining,
                mode='lines', line=dict(color='#9E9E9E', width=1, dash='dash'),
                name='剩余信号'
            ))

        frames.append(go.Frame(data=data, name=f'提取 {labels[step]}'))

    # 最终帧: 重构验证
    recon = sum(components)
    final_data = [
        go.Scatter(x=t, y=signal, mode='lines',
                   line=dict(color='#333', width=1.5), name='原始信号'),
        go.Scatter(x=t, y=recon, mode='lines',
                   line=dict(color='red', width=1, dash='dash'), name='重构 (IMF之和)'),
    ]
    for j in range(len(components)):
        offset = -(j + 1) * 2.5
        final_data.append(go.Scatter(
            x=t, y=components[j] + offset,
            mode='lines', line=dict(color=colors[j], width=1.2),
            name=labels[j]
        ))
    frames.append(go.Frame(data=final_data, name='重构验证: sum(IMFs) + Residual'))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title=dict(text='EMD 经验模态分解动画 (信号 -> IMFs + Residual)'),
        xaxis_title='时间', yaxis_title='幅值',
        height=600, width=1000,
        plot_bgcolor='white',
        updatemenus=[dict(type='buttons', x=0.1, y=1.08, buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 1500}, 'transition': {'duration': 500}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0, x=0.05, len=0.9,
        )],
    )
    return fig

fig = animate_emd_decomposition()
fig.show()

In [ ]:
# ============================================================
# Cell 4: 导入与环境设置
# ============================================================
import sys, os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from shared.data_fetcher import get_stock_daily
from shared.backtest_engine import Backtester
from shared.visualization import plot_full_report, set_chinese_font

set_chinese_font()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

# EMD - 尝试 emd-signal 或回退到简单实现
try:
    from emd import sift
    HAS_EMD = True
    print('使用 emd-signal 库进行EMD分解')
except ImportError:
    HAS_EMD = False
    print('emd-signal 未安装，使用内置简化EMD实现')
    print('安装: pip install emd-signal')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
SYMBOL = '000858'  # 五粮液
df = get_stock_daily(SYMBOL, start_date='20200601', end_date='20241231')
print(f'数据形状: {df.shape}')
print(f'日期范围: {df.index[0]} ~ {df.index[-1]}')
df.tail(3)

In [ ]:
# ============================================================
# Cell 6: 特征工程 - EMD分解
# ============================================================
LOOKBACK = 30
MAX_IMFS = 5  # 最多保留的IMF分量数

def simple_emd(signal, max_imfs=5, max_siftings=100):
    """
    简化版EMD实现 (当emd-signal不可用时的备选)
    使用scipy的三次样条插值实现筛分过程
    """
    from scipy.interpolate import CubicSpline
    from scipy.signal import argrelextrema

    imfs = []
    residual = signal.copy()

    for _ in range(max_imfs):
        h = residual.copy()
        for _ in range(max_siftings):
            # 找极值点
            maxima = argrelextrema(h, np.greater)[0]
            minima = argrelextrema(h, np.less)[0]

            if len(maxima) < 2 or len(minima) < 2:
                break

            # 上下包络线
            t = np.arange(len(h))
            try:
                upper_env = CubicSpline(maxima, h[maxima], extrapolate=True)(t)
                lower_env = CubicSpline(minima, h[minima], extrapolate=True)(t)
            except Exception:
                break

            mean_env = (upper_env + lower_env) / 2
            h = h - mean_env

            # 检查收敛
            if np.mean(mean_env ** 2) < 1e-10 * np.mean(signal ** 2):
                break

        imfs.append(h)
        residual = residual - h

        # 如果残差太小则停止
        if np.std(residual) < 1e-6 * np.std(signal):
            break

    imfs.append(residual)  # 最后一个是残差
    return np.array(imfs)


def emd_decompose(signal, max_imfs=5):
    """执行EMD分解"""
    if HAS_EMD:
        imfs_matrix = sift.sift(signal, max_imfs=max_imfs)
        # emd-signal返回 (n_samples, n_imfs)
        imfs = [imfs_matrix[:, i] for i in range(imfs_matrix.shape[1])]
    else:
        imfs_matrix = simple_emd(signal, max_imfs=max_imfs)
        imfs = [imfs_matrix[i] for i in range(imfs_matrix.shape[0])]
    return imfs


# 对收盘价序列进行EMD分解
close_prices = df['close'].values.astype(float)
imfs = emd_decompose(close_prices, max_imfs=MAX_IMFS)
n_imfs = len(imfs)

print(f'EMD分解得到 {n_imfs} 个分量:')
for i, imf in enumerate(imfs):
    label = f'IMF {i+1}' if i < n_imfs - 1 else 'Residual'
    print(f'  {label}: std={np.std(imf):.4f}, shape={imf.shape}')

# 验证重构
recon = sum(imfs)
recon_error = np.mean((close_prices - recon[:len(close_prices)]) ** 2)
print(f'\n重构误差 (MSE): {recon_error:.6f}')

In [ ]:
# ============================================================
# Cell 7: PyTorch EMD-LSTM 模型与训练
# ============================================================

class SingleIMFLSTM(nn.Module):
    """单个IMF分量的LSTM预测模型"""
    def __init__(self, hidden_dim=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        """x: (batch, seq_len, 1) -> (batch, 1)"""
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class EMDLSTM(nn.Module):
    """EMD-LSTM: 为每个IMF分量训练独立LSTM，合并预测"""
    def __init__(self, n_imfs, hidden_dim=32):
        super().__init__()
        self.n_imfs = n_imfs
        self.branches = nn.ModuleList([
            SingleIMFLSTM(hidden_dim) for _ in range(n_imfs)
        ])

    def forward(self, imf_inputs):
        """imf_inputs: list of (batch, seq_len, 1)"""
        predictions = [branch(inp) for branch, inp in zip(self.branches, imf_inputs)]
        # 求和合成
        return sum(predictions)


# --- 构建训练数据 ---
def build_emd_dataset(imfs, close_prices, lookback=30):
    """为每个IMF分量构建滑窗序列"""
    n = len(close_prices)
    n_imfs = len(imfs)

    X_imfs = [[] for _ in range(n_imfs)]  # 输入: 各IMF历史窗口
    y_list = []  # 标签: 下一时刻的真实价格（用于回归）
    y_dir = []   # 方向标签
    dates_out = []

    for i in range(lookback, n - 1):
        for k in range(n_imfs):
            X_imfs[k].append(imfs[k][i-lookback:i].reshape(-1, 1))
        # 标签: 下一时刻的价格
        y_list.append(close_prices[i + 1])
        # 方向: 涨(1) / 跌(0)
        y_dir.append(1 if close_prices[i + 1] > close_prices[i] else 0)
        dates_out.append(df.index[i])

    X_arrays = [np.array(x, dtype=np.float32) for x in X_imfs]
    y_price = np.array(y_list, dtype=np.float32)
    y_direction = np.array(y_dir, dtype=np.float32)
    return X_arrays, y_price, y_direction, dates_out


X_imf_arrays, y_price, y_dir, dates = build_emd_dataset(imfs, close_prices, LOOKBACK)

# --- 数据划分 ---
TRAIN_END = pd.Timestamp('2023-12-31')
dates_ts = pd.DatetimeIndex(dates)
train_mask = dates_ts <= TRAIN_END
test_mask = dates_ts > TRAIN_END

# 标准化每个IMF
scalers_imf = []
X_train_imfs, X_test_imfs = [], []
for k in range(n_imfs):
    sc = StandardScaler()
    train_flat = X_imf_arrays[k][train_mask].reshape(-1, 1)
    sc.fit(train_flat)
    scalers_imf.append(sc)

    train_s = sc.transform(X_imf_arrays[k][train_mask].reshape(-1, 1)).reshape(-1, LOOKBACK, 1)
    test_s = sc.transform(X_imf_arrays[k][test_mask].reshape(-1, 1)).reshape(-1, LOOKBACK, 1)
    X_train_imfs.append(torch.FloatTensor(train_s))
    X_test_imfs.append(torch.FloatTensor(test_s))

# 价格标签标准化
price_scaler = StandardScaler()
y_train_price = price_scaler.fit_transform(y_price[train_mask].reshape(-1, 1)).flatten()
y_test_price = y_price[test_mask]
y_test_dir = y_dir[test_mask]
test_dates_out = dates_ts[test_mask]

y_train_t = torch.FloatTensor(y_train_price)
print(f'训练: {len(y_train_t)} 样本, 测试: {len(y_test_dir)} 样本')
print(f'IMF分量数: {n_imfs}')

# DataLoader
train_dataset = TensorDataset(*X_train_imfs, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 训练 ---
model = EMDLSTM(n_imfs=n_imfs, hidden_dim=32).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_s = 0
    for batch in train_loader:
        imf_inputs = [b.to(device) for b in batch[:n_imfs]]
        labels = batch[n_imfs].to(device)

        optimizer.zero_grad()
        pred = model(imf_inputs).squeeze(-1)
        loss = criterion(pred, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(labels)
        n_s += len(labels)
    scheduler.step()

    avg_loss = epoch_loss / max(n_s, 1)
    train_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}  MSE Loss: {avg_loss:.6f}')

# --- 预测 ---
model.eval()
with torch.no_grad():
    test_inputs = [x.to(device) for x in X_test_imfs]
    pred_scaled = model(test_inputs).cpu().numpy().flatten()

# 反标准化得到预测价格
pred_prices = price_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()

# 方向判断: 预测价格 > 当前价格 -> 涨
current_prices = close_prices[test_mask.nonzero()[0]]
if len(current_prices) > len(pred_prices):
    current_prices = current_prices[:len(pred_prices)]
test_preds_dir = (pred_prices > current_prices).astype(int)

accuracy = (test_preds_dir == y_test_dir[:len(test_preds_dir)]).mean()
print(f'\n方向预测准确率: {accuracy:.4f}')

In [ ]:
# ============================================================
# Cell 8: 回测
# ============================================================
n_test = min(len(test_preds_dir), len(test_dates_out))
signals = pd.Series(
    test_preds_dir[:n_test],
    index=test_dates_out[:n_test],
    name='signal'
)
test_prices_bt = df.loc[test_dates_out[:n_test], 'close']

bt = Backtester(initial_capital=1_000_000, t_plus_1=True)
result = bt.run(test_prices_bt, signals)

print('回测绩效指标:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. EMD分解图
n_show = min(n_imfs, 6)
fig, axes = plt.subplots(n_show + 1, 1, figsize=(14, 3 * (n_show + 1)), sharex=True)

axes[0].plot(df.index, close_prices, color='#333', linewidth=1)
axes[0].set_title('原始收盘价', fontsize=11)
axes[0].grid(True, alpha=0.3)

colors = ['#F44336', '#FF9800', '#FFC107', '#4CAF50', '#2196F3', '#9C27B0']
for i in range(n_show):
    label = f'IMF {i+1}' if i < n_imfs - 1 else 'Residual'
    axes[i+1].plot(df.index, imfs[i][:len(df)], color=colors[i % len(colors)], linewidth=1)
    axes[i+1].set_title(label, fontsize=11)
    axes[i+1].grid(True, alpha=0.3)

fig.suptitle('EMD 经验模态分解结果', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 2. 训练损失
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, color='#2196F3', linewidth=1.5)
ax.set_title('EMD-LSTM 训练损失 (MSE)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. 价格预测 vs 实际
n_plot = min(100, len(pred_prices))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates_out[:n_plot], y_test_price[:n_plot],
        color='#333', linewidth=1.5, label='实际价格')
ax.plot(test_dates_out[:n_plot], pred_prices[:n_plot],
        color='#F44336', linewidth=1.2, linestyle='--', label='预测价格')
ax.set_title('EMD-LSTM 价格预测 vs 实际', fontsize=14, fontweight='bold')
ax.set_xlabel('日期')
ax.set_ylabel('价格')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 4. 完整回测报告
plot_full_report(result, strategy_name='EMD-LSTM (Chen 2024)')

## 结果分析

### 模型架构

- **EMD分解**: 将价格序列自适应分解为多个IMF + 残差
- **独立LSTM**: 每个IMF分量由独立LSTM模型预测
- **预测合成**: 各分量预测值求和 -> 与当前价格比较得出方向
- **回归 -> 方向**: 先预测价格（回归），再转化为涨跌方向

### EMD vs 小波分解 (Notebook 08对比)

| 特性 | EMD | DWT (小波) |
|------|-----|------------|
| 基函数 | 自适应 (数据驱动) | 预定义 (db4等) |
| 分解方式 | 筛分迭代 | 滤波器组 |
| 频率分辨率 | 瞬时频率 | 固定频段 |
| 端点效应 | 较严重 | 较轻微 |
| 适用场景 | 非线性非平稳信号 | 一般信号 |

### 关键发现

1. EMD的自适应分解无需预选基函数，更适合金融时序的非平稳特性
2. 分量级别的独立建模避免了不同频率信息的相互干扰
3. 回归方式预测价格再转方向，比直接分类可能更稳定

### 局限性

- EMD存在模态混叠(mode mixing)问题，EEMD/CEEMDAN等变体可改善
- 筛分过程对端点敏感，新数据点加入可能改变已有分解
- 论文报告的71-127%年化收益率可能存在过拟合风险